# Backend API Smoke Test\n
\n
Start the backend first (`yarn dev` inside `backend`).

In [26]:
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from IPython.display import HTML, display

BASE_URL = "http://localhost:3000"


In [ ]:
def get_json(path: str):
    req = Request(f"{BASE_URL}{path}", method="GET")
    try:
        with urlopen(req, timeout=30) as resp:
            body = resp.read().decode("utf-8")
            return resp.status, json.loads(body)
    except HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        return e.code, {"error": body}
    except URLError as e:
        return None, {"error": str(e)}

def post_json(path: str, payload: dict):
    data = json.dumps(payload).encode("utf-8")
    req = Request(
        f"{BASE_URL}{path}",
        data=data,
        method="POST",
        headers={"Content-Type": "application/json"},
    )
    try:
        with urlopen(req, timeout=300) as resp:  # 5 minutes
            body = resp.read().decode("utf-8")
            return resp.status, json.loads(body)
    except HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        return e.code, {"error": body}
    except URLError as e:
        return None, {"error": str(e)}

In [28]:
def render_html(html: str, width: int = 800, height: int = 800):
    iframe = f'''<iframe srcdoc="{html.replace(chr(34), '&quot;')}" width="{width}" height="{height}" style="border:1px solid #ccc; border-radius:8px;"></iframe>'''
    display(HTML(iframe))


In [29]:
# 1) Health check
status, data = get_json("/")
print("status:", status)
print(json.dumps(data, indent=2, ensure_ascii=False))

status: 200
{
  "ok": true,
  "message": "Backend is running"
}


In [ ]:
# 2) Generic generation route
status, data = post_json("/ai/gen_engine", {"prompt": "Write one short test sentence."})
print("status:", status)
print(json.dumps(data, indent=2, ensure_ascii=False)[:1500])

In [31]:
# 3) Webgame HTML generation route
payload = {
    "gameDescription": "NEON RONIN — The Cat's Labyrinth — A reimagined Pac-Man set in a cyberpunk Tokyo at night. The character is a robot-ronin chasing a ghost-cat through neon-lit alleyways. The \"ghosts\" are corrupted samurai with movement patterns inspired by Dark Souls bosses. Procedurally generated lo-fi jazz soundtrack. Permadeath (roguelike) with a layout that changes every run. Base: Pac-Man — Theme: Neon Tokyo — Boss: Dark Souls — Audio: Lo-fi jazz — Mechanic: Roguelike",
    "size": 720
}

status, data = post_json("/ai/gen_engine/webgame", payload)
print("status:", status)
if isinstance(data, dict) and "html" in data:
    html = data["html"]
    print("HTML length:", len(html))
    print("Model:", data.get("model"))
    render_html(html, width=760, height=760)
else:
    print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])


TimeoutError: timed out